# RepoVeritas v1 — Qwen2.5 baseline sweep

Evaluates **Qwen2.5-{0.5B, 1.5B, 3B, 7B}-Instruct** (7B in 4-bit for a T4) on the held-out `test` split,
scoring the three-way grounding task (supported / contradicted / insufficient).

**Headline metric: insufficient-F1** (the abstention class). Floors from the harness:
`majority` (always "supported") = **0.000**, `random` = **0.188**. A model has to clear those to show it
can do the hard judgment at all.

**Outputs:** results table (accuracy, macro-F1, supported/contradicted/insufficient-F1), a confusion matrix
per model, per-family (docstring vs commit) breakdown, an insufficient-F1-vs-scale chart, and saved
predictions (JSONL + CSV) for failure-case inspection.

**Runtime:** ~45–60 min for the full 207-item test set across all four models on a free T4. Set `SMOKE`
in the load cell to ~12 first for a 2-minute sanity check.

> Runtime → Change runtime type → **GPU (T4)** before running.

In [ ]:
!pip install -q -U transformers accelerate bitsandbytes

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0),
          f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

## 1. Upload the release files
Upload **`baseline_eval.py`**, **`test_public.jsonl`**, and **`test_gold.jsonl`** (from `release_v1/`).

In [ ]:
try:
    from google.colab import files
    print("Select baseline_eval.py, test_public.jsonl, test_gold.jsonl ...")
    files.upload()
except Exception as e:
    print("Not in Colab? Put the three files next to this notebook.", e)

In [ ]:
import json
from collections import Counter
from baseline_eval import build_prompt, parse_label, score, fam_f1, LABELS

PUBLIC = [json.loads(l) for l in open("test_public.jsonl") if l.strip()]
GOLD = {json.loads(l)["public_id"]: json.loads(l) for l in open("test_gold.jsonl") if l.strip()}

SMOKE = 0   # set to ~12 for a quick sanity run; 0 = full test set
items = PUBLIC[:SMOKE] if SMOKE else PUBLIC
print(len(items), "items |", dict(Counter(GOLD[i["id"]]["label"] for i in items)))

## 2. Run the Qwen2.5 sweep
Each model is loaded, evaluated, then freed before the next (one model fits the T4 at a time; 7B uses 4-bit nf4).

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm.auto import tqdm
import gc

MODELS = [
    ("0.5B", "Qwen/Qwen2.5-0.5B-Instruct", 0.5, False),
    ("1.5B", "Qwen/Qwen2.5-1.5B-Instruct", 1.5, False),
    ("3B",   "Qwen/Qwen2.5-3B-Instruct",   3.0, False),
    ("7B",   "Qwen/Qwen2.5-7B-Instruct",   7.0, True),   # 4-bit for T4
]
MAX_NEW = 200   # brief reasoning + the Label line

def run_model(model_id, four_bit):
    tok = AutoTokenizer.from_pretrained(model_id)
    kw = dict(torch_dtype=torch.float16, device_map="auto")
    if four_bit:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    model = AutoModelForCausalLM.from_pretrained(model_id, **kw).eval()
    preds = []
    for it in tqdm(items, desc=model_id.split("/")[-1]):
        msgs = [{"role": "user", "content": build_prompt(it)}]
        text = tok.apply_chat_template(msgs, add_generation_prompt=True, tokenize=False)
        enc = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=MAX_NEW, do_sample=False,
                                 pad_token_id=tok.eos_token_id)
        raw = tok.decode(out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True)
        g = GOLD[it["id"]]
        preds.append({"id": it["id"], "task_family": it["task_family"],
                      "gold": g["label"], "pred": parse_label(raw), "raw": raw})
    del model, tok; gc.collect(); torch.cuda.empty_cache()
    return preds

results, all_preds = {}, {}
for tag, mid, params, fb in MODELS:
    print(f"\n=== Qwen2.5-{tag}  (4-bit={fb}) ===")
    preds = run_model(mid, fb); all_preds[tag] = preds
    rows = [(p["gold"], p["pred"], p["task_family"]) for p in preds]
    cm, per, acc, macro = score(rows)
    results[tag] = dict(params=params, accuracy=acc, macro_f1=macro,
        supported_f1=per["supported"][2], contradicted_f1=per["contradicted"][2],
        insufficient_f1=per["insufficient"][2], cm=cm, rows=rows,
        unparseable=sum(1 for _, p, _ in rows if p not in LABELS))
    print(f"  acc={acc:.3f}  macro-F1={macro:.3f}  >>> insufficient-F1={per['insufficient'][2]:.3f} <<<"
          f"  (unparsed={results[tag]['unparseable']})")

## 3. Results table

In [ ]:
import pandas as pd
df = pd.DataFrame([dict(model=f"Qwen2.5-{t}", params_B=r["params"],
    accuracy=round(r["accuracy"], 3), macro_F1=round(r["macro_f1"], 3),
    supported_F1=round(r["supported_f1"], 3), contradicted_F1=round(r["contradicted_f1"], 3),
    insufficient_F1=round(r["insufficient_f1"], 3), unparseable=r["unparseable"])
    for t, r in results.items()])
df.to_csv("repoveritas_baseline_results.csv", index=False)
print("HEADLINE = insufficient_F1   (floors: majority 0.000, random 0.188)")
df

## 4. Confusion matrices (rows = gold, cols = pred)
Watch the **insufficient → supported** cell: that's the over-claiming failure mode.

In [ ]:
import matplotlib.pyplot as plt, numpy as np
cols = LABELS + ["??"]
n = len(results)
fig, axes = plt.subplots(1, n, figsize=(4.2*n, 3.8))
if n == 1: axes = [axes]
for ax, (t, r) in zip(axes, results.items()):
    M = np.array([[r["cm"].get((g, c), 0) for c in cols] for g in LABELS])
    ax.imshow(M, cmap="Blues")
    ax.set_xticks(range(len(cols))); ax.set_xticklabels([c[:4] for c in cols], fontsize=8)
    ax.set_yticks(range(len(LABELS))); ax.set_yticklabels(LABELS, fontsize=8)
    ax.set_title(f"Qwen2.5-{t}"); ax.set_xlabel("pred"); ax.set_ylabel("gold")
    for i in range(len(LABELS)):
        for j in range(len(cols)):
            ax.text(j, i, int(M[i, j]), ha="center", va="center", fontsize=8,
                    color="white" if M[i, j] > M.max()/2 else "black")
plt.tight_layout(); plt.savefig("repoveritas_confusion.png", dpi=130); plt.show()

## 5. Headline chart — insufficient-F1 vs model scale

In [ ]:
xs = [r["params"] for r in results.values()]
plt.figure(figsize=(7.5, 5))
plt.plot(xs, [r["insufficient_f1"] for r in results.values()], "o-", lw=2.2, label="insufficient-F1 (headline)")
plt.plot(xs, [r["macro_f1"] for r in results.values()], "s--", color="gray", label="macro-F1")
plt.plot(xs, [r["supported_f1"] for r in results.values()], "^:", color="green", alpha=0.7, label="supported-F1")
plt.axhline(0.188, color="orange", ls=":", label="random floor (insuf-F1)")
plt.xscale("log"); plt.xticks(xs, [str(x) for x in xs])
plt.xlabel("Qwen2.5 size (B params, log scale)"); plt.ylabel("F1")
plt.title("RepoVeritas: abstention (insufficient) F1 vs model scale")
plt.legend(); plt.grid(alpha=0.3); plt.ylim(0, 1); plt.tight_layout()
plt.savefig("repoveritas_insufficient_f1_vs_scale.png", dpi=130); plt.show()

## 6. Save predictions + per-family breakdown + failure cases

In [ ]:
import csv
for t, preds in all_preds.items():
    with open(f"preds_qwen{t}.jsonl", "w") as f:
        for p in preds: f.write(json.dumps(p) + "\n")
with open("repoveritas_all_predictions.csv", "w", newline="") as f:
    w = csv.writer(f); w.writerow(["model", "id", "task_family", "gold", "pred", "correct", "raw_first240"])
    for t, preds in all_preds.items():
        for p in preds:
            w.writerow([t, p["id"], p["task_family"], p["gold"], p["pred"],
                        int(p["pred"] == p["gold"]), (p["raw"] or "")[:240].replace(chr(10), " ")])
print("saved: repoveritas_baseline_results.csv, repoveritas_confusion.png,",
      "repoveritas_insufficient_f1_vs_scale.png, preds_qwen*.jsonl, repoveritas_all_predictions.csv")

# per-family insufficient-F1 across models
print("\nper-family insufficient-F1:")
for t, r in results.items():
    d = fam_f1(r["rows"], "insufficient", "docstring_function")
    c = fam_f1(r["rows"], "insufficient", "commit_diff")
    print(f"  Qwen2.5-{t}:  docstring={d:.3f}  commit={c:.3f}")

# inspect the over-claiming failure mode on the largest model
t = list(all_preds)[-1]
errs = [p for p in all_preds[t] if p["gold"] == "insufficient" and p["pred"] == "supported"]
print(f"\n[Qwen2.5-{t}] {len(errs)} insufficient->supported errors (over-claiming). Examples:")
for p in errs[:6]:
    print("  ", p["id"], "|", (p["raw"] or "")[:150].replace(chr(10), " "))

# to download everything:
# from google.colab import files
# for fn in ["repoveritas_baseline_results.csv","repoveritas_confusion.png",
#            "repoveritas_insufficient_f1_vs_scale.png","repoveritas_all_predictions.csv"]:
#     files.download(fn)

## Adding a frontier datapoint (optional)
For a ceiling on the range, evaluate one API model with the *same* prompt via the harness adapters:

```python
from baseline_eval import make_model_fn, build_prompt, parse_label, score
import os; os.environ["ANTHROPIC_API_KEY"] = "sk-..."   # your key
fn = make_model_fn("anthropic:claude-3-5-sonnet-latest")   # or openai:gpt-4o
rows = []
for it in items:
    raw = fn(build_prompt(it)); g = GOLD[it["id"]]
    rows.append((g["label"], parse_label(raw), it["task_family"]))
cm, per, acc, macro = score(rows)
print("frontier insufficient-F1:", round(per["insufficient"][2], 3), "| macro-F1:", round(macro, 3))
```

**What to look for:** supported-F1 high across all sizes, **insufficient-F1 well below it** and rising slowly
(or flat) with scale, and the confusion mass sitting in **insufficient→supported**. That gap, on a
human-verified set, is the result.